# Constraint-Induced Factor Alignment Problems

This notebook studies a portfolio-construction problem that can arise when an optimizer combines an alpha model, a risk model, and investment constraints. It also serves as a small demonstration of how to utilize the package and show its functionalities.

In mean-variance optimization, the portfolio holdings $h$ are chosen by solving

$$
\max_h \ \alpha^\top h - \frac{\lambda}{2} h^\top Q h,
$$

subject to constraints such as full investment, long-only restrictions, and active asset bounds. Here, $h$ is the vector of portfolio weights, $\alpha$ is the alpha signal or expected return forecast, $Q$ is the covariance matrix used by the optimizer to measure risk, and $\lambda$ controls risk aversion. The optimizer therefore tries to find a portfolio with high expected return $\alpha^\top h$ and low model-implied variance $h^\top Qh$.

The potential problem is that the covariance matrix $Q$ is only a model. It may capture the main risk factors, but it does not necessarily contain every systematic risk direction that exists in the true return process. If the optimizer loads onto a direction that is missing from $Q$, the portfolio may look safe according to the model while being riskier in reality. This is the basic idea behind a factor alignment problem.

A factor risk model assumes that asset returns are partly driven by a small number of common factors. In this notebook, those factors are collected in an exposure matrix $X$, where each row corresponds to an asset and each column corresponds to a risk factor. Commercial equity risk models often contain factors such as market, country, industry, value, size, momentum, volatility, or quality. The exact list is not important here. What matters is that the model only spans a limited factor space.

Any asset-level vector $z$ can be decomposed into a part that is explained by the model factors and a residual part that is not:

$$
z = z_{\text{X}} + z_{\perp}.
$$

The spanned component $z_{\text{X}}$ lies in the column space of $X$. The orthogonal component $z_\perp$ is the residual after projecting $z$ onto the factor space. In the code, this is done by the projection function. When we write $\alpha_\perp$, we mean the part of the alpha vector that is not explained by the risk-model factors. If $\alpha_\perp = 0$, the alpha signal is fully aligned with the risk model. If $\alpha_\perp \neq 0$, then part of the alpha signal points in a direction that the risk model does not represent.

This matters because optimizers respond mechanically to the risk model they are given. If a return direction is represented in $Q$, the optimizer sees that holding it creates systematic risk and charges a risk penalty. If a direction is orthogonal to the modelled risk factors, the optimizer does not see it as a systematic factor exposure. It may still carry specific risk, but it is not treated as a common risk factor. As a result, an unmodelled direction can appear unusually attractive: it may offer expected return without much model-implied systematic risk. The optimizer can therefore load too heavily onto it.

The standard version of this problem occurs when the raw alpha itself has an unmodelled component, meaning $\alpha_\perp \neq 0$. This notebook focuses on a subtler case. We deliberately construct alpha so that it is fully explained by the risk model: $\alpha_\perp \approx 0.$

At first, this appears safe: the alpha and risk model are aligned. However, the portfolio is constrained. The constrained optimizer has inequality constraints that can be written as $Ah \leq u.$

The shadow prices of these constraints are denoted by $\pi$. A shadow price measures how much a constraint affects the optimal solution. If a constraint is not binding, its shadow price is zero. If it is binding, it changes the optimizer's effective tradeoff.

The key object is the implied alpha,

$$
\gamma = \alpha - A^\top \pi.
$$

This is the alpha signal after accounting for the effect of binding constraints. Even if the original alpha is fully spanned by the risk model, the implied alpha need not be. Constraints can tilt the effective alpha into a direction that the risk model does not span. In other words, it is possible to have

$$
\alpha_\perp \approx 0
\quad \text{but} \quad
\gamma_\perp \neq 0.
$$

This is the constraint-induced factor alignment problem studied in this notebook.

The intuition is easiest to see through active bounds. Suppose a manager has an alpha signal that is completely represented in the risk model. The optimizer wants to buy the highest-alpha assets and sell the lowest-alpha assets. If the portfolio is long-only and has upper active bounds, the highest-alpha assets may hit their upper bounds, while the lowest-alpha assets may hit their lower bounds. The binding constraints are then concentrated in the top and bottom alpha tails.

This creates the following chain:

$$
\text{top/bottom alpha assets}
\quad \Rightarrow \quad
\text{binding constraints}
\quad \Rightarrow \quad
\text{constraint pressure}
\quad \Rightarrow \quad
\gamma_\perp.
$$

If those top and bottom alpha assets share a systematic characteristic that is missing from the risk model, the optimizer can accidentally create exposure to a hidden risk factor. Importantly, this can happen even though the original alpha vector was fully aligned with the model.

The Alpha Alignment Factor, or AAF, is a practical way to address this. It adds one auxiliary risk factor based on the orthogonal part of implied alpha:

$$
\hat y = \frac{\gamma_\perp}{\|\gamma_\perp\|}.
$$

The purpose of $\hat y$ is to represent the unmodelled direction that the constrained optimizer is being pushed toward. The corresponding augmented covariance matrix is

$$
Q_{\text{AAF}}
=
Q_{\text{model}}
+
\hat{\nu} \hat y \hat y^\top,
$$

where $\hat{\nu}$ is the estimated variance of the latent factor return associated with $\hat y$. The optimizer then solves the same portfolio problem with the same alpha and the same constraints, but using $Q_{\text{AAF}}$ instead of $Q_{\text{model}}$.

The goal is not to remove risk from the portfolio. The goal is to make the optimizer recognize risk that was previously missing from its model. If the base optimizer was taking an unintended hidden-factor bet, the AAF optimizer should reduce that exposure and produce a risk estimate closer to the true risk.

In [ ]:
import numpy as np

from factoralign.constraints import fully_invested, active_bounds
from factoralign.optimizer import solve_mvo
from factoralign.projection import project_onto_factors, normalize_vector
from factoralign.alignment import analyze_alignment
from factoralign.latent import estimate_latent_factor_returns, estimate_latent_variance
from factoralign.aaf import augment_covariance, solve_aaf_mvo


def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


def var(h, Q):
    return float(h @ Q @ h)

We create a universe of 10 assets. The benchmark is equal-weighted, so each asset has benchmark weight $b_i = 0.10.$

The exposure matrix $X$ has two columns:

1. a common market-like factor, represented by a column of ones;
2. a style factor $x$, increasing from $-1$ to $1$.

The alpha signal is $\alpha = 0.14x.$ This means alpha is exactly explained by the risk model, because $x$ is one of the columns of $X$. The covariance matrix has factor-model form

$$
Q_{\text{model}} = XFX^\top + D.
$$

This gives the optimizer a simple risk model with common factor risk and asset-specific risk.

In [ ]:
rng = np.random.default_rng(1)


n = 10
b = np.full(n, 1 / n) 

x = np.linspace(-1, 1, n) 
X = np.column_stack([np.ones(n), x]) 

alpha = 0.14 * x 

risk_aversion = 2.0
upper_active = 1 / n          

factor_cov = np.array([
    [0.04, 0.00],             
    [0.00, 0.005],            
])

specific_var = 0.35 + 0.10 * np.abs(x)
Q_model = X @ factor_cov @ X.T + np.diag(specific_var)

weights = np.ones(n)

The portfolio must satisfy $\sum_i h_i = 1.$ It is also long-only: $h_i \geq 0.$ Finally, each asset has an active upper bound: $h_i - b_i \leq 0.10.$ Since $b_i = 0.10$, this implies $h_i \leq 0.20.$ So each asset can be held between 0% and 20%.

These constraints are simple, but they are enough to create the mechanism we want to study. The optimizer will try to sell low-alpha assets and buy high-alpha assets, but the bounds restrict how far it can go.

In [ ]:
constraints = fully_invested(n).combine(
    active_bounds(
        benchmark=b,
        lower_active=-b,
        upper_active=np.full(n, upper_active),
    )
)

We now define the top and bottom alpha tails. The bottom tail contains the two assets with the lowest alpha. The top tail contains the two assets with the highest alpha. We create a vector $z_{\text{tail}}$ that keeps alpha only for these tail assets and sets all middle assets to zero:

$$
z_{\text{tail}, i}
=
\begin{cases}
\alpha_i, & i \in \text{top or bottom tail}, \\
0, & \text{otherwise}.
\end{cases}
$$

This vector captures the structure of the assets most likely to hit the active bounds. The intuition is this: imagine the highest predicted assets all are small companies and the lowest predicted are large companies. Then the tail vector captures a small-minus-big style factor. If the risk model does not contain a size factor, this tail vector is a good candidate for the hidden factor that the optimizer is being pushed toward.

Even though $\alpha$ itself is fully spanned by $X$, the tail vector $z_{\text{tail}}$ is not necessarily spanned by $X$. It is a thresholded version of alpha, not a smooth linear factor. We define the true hidden factor as the normalized orthogonal component of this tail vector:

$$
y_{\text{true}}
=
\frac{P_\perp(z_{\text{tail}})}
{\|P_\perp(z_{\text{tail}})\|}.
$$

In [142]:
n_tail = 2
order = np.argsort(alpha)
bottom = order[:n_tail]
top = order[-n_tail:]

z_tail = np.zeros(n)
z_tail[bottom] = alpha[bottom]
z_tail[top] = alpha[top]

alpha_proj = project_onto_factors(alpha, X, weights)
tail_proj = project_onto_factors(z_tail, X, weights)
y_true = normalize_vector(tail_proj.orthogonal)

The base optimizer uses the incomplete risk model $Q_{\text{model}}$. It solves

$$
\max_h \ \alpha^\top h - \frac{\lambda}{2}h^\top Q_{\text{model}}h
$$

subject to the portfolio constraints. We then inspect which constraints are binding. The desired result is that the highest-alpha assets hit the upper bound and the lowest-alpha assets hit the lower bound. If this happens, the active constraints are concentrated exactly in the alpha tails.

In [143]:
base = solve_mvo(
    alpha=alpha,
    covariance=Q_model,
    constraints=constraints,
    risk_aversion=risk_aversion,
)

h_base = base.holdings

After solving the optimizer, we compute the constraint pressure $A^\top \pi.$ This is the part of the optimizer's effective signal that comes from binding constraints.

The implied alpha is $\gamma = \alpha - A^\top \pi.$ Since $\alpha_\perp \approx 0$, any orthogonal component in $\gamma$ must come from the constraints.

We estimate the AAF direction as

$$
\hat y = \frac{\gamma_\perp}{\|\gamma_\perp\|}.
$$

We then compare $\hat y$ with the true hidden direction $y_{\text{true}}$. If their cosine is high, then the AAF direction is capturing the hidden tail-risk direction created by the active constraints.

In [144]:
analysis = analyze_alignment(
    alpha=alpha,
    constraints=constraints,
    ineq_duals=base.ineq_duals,
    eq_duals=base.eq_duals,
    exposures=X,
    weights=weights,
)

p = analysis.pressure_ineq.orthogonal
y_hat = analysis.gamma_direction(include_equalities=False)

if y_hat @ y_true < 0:
    y_hat = -y_hat

binding = constraints.binding_mask(h_base, tol=1e-7)
upper_bind = binding[:n]
lower_bind = binding[n:2*n]

print("alpha orth share:", np.linalg.norm(alpha_proj.orthogonal) / np.linalg.norm(alpha))
print("tail orth share:", np.linalg.norm(tail_proj.orthogonal) / np.linalg.norm(z_tail))

print("\nholdings:", np.round(h_base, 4))
print("upper binding assets:", np.where(upper_bind)[0])
print("lower binding assets:", np.where(lower_bind)[0])
print("top tail:", top)
print("bottom tail:", bottom)

print("\npressure orth norm:", np.linalg.norm(p))
print("cos(pressure_orth, y_true):", cos(p, y_true))
print("cos(y_hat, y_true):", cos(y_hat, y_true))

alpha orth share: 4.146344666145215e-16
tail orth share: 0.4605661864718382

holdings: [ 0.     -0.      0.0019  0.0411  0.0851  0.1267  0.1585  0.1868  0.2
  0.2   ]
upper binding assets: [8 9]
lower binding assets: [0 1]
top tail: [8 9]
bottom tail: [0 1]

pressure orth norm: 0.03976592325075878
cos(pressure_orth, y_true): 0.7769205412589846
cos(y_hat, y_true): 0.7769205412589849



The base optimizer uses $Q_{\text{model}}$, but the simulated true world contains an additional hidden factor:

$$
Q_{\text{true}}
=
Q_{\text{model}}
+
\nu_{\text{true}} y_{\text{true}} y_{\text{true}}^\top.
$$

This hidden factor is not available to the base optimizer. If the base portfolio has exposure to $y_{\text{true}}$, then its true variance exceeds its model variance by $\nu_{\text{true}} \left(h_{\text{base}}^\top y_{\text{true}}\right)^2.$ This is why the base risk model can underestimate optimized portfolio risk.

To estimate the variance of the missing factor, we simulate asset returns from the true model:

$$
r_t = Xf_t + y_{\text{true}}g_t + \epsilon_t.
$$

The estimation procedure does not use $y_{\text{true}}$. Instead, it uses $\hat y$, the direction found from implied alpha. For every time period, we run a cross-sectional regression:

$$
r_t = Xf_t + \hat y \hat{g}_t + e_t.
$$

This gives an estimated latent factor return $\hat{g}_t$. The sample variance of this return series is $\hat \nu$

In [145]:
nu_true = 10.0
Q_true = Q_model + nu_true * np.outer(y_true, y_true)

print("\nbase risk:")
print(f"base model variance: {var(h_base, Q_model):.8f}")
print(f"base true variance:  {var(h_base, Q_true):.8f}")
print("base exposure y_true:", float(h_base @ y_true))

T = 5000

factor_returns = rng.multivariate_normal(
    mean=np.zeros(X.shape[1]),
    cov=factor_cov,
    size=T,
)

latent_true_returns = rng.normal(
    scale=np.sqrt(nu_true),
    size=T,
)

specific_returns = rng.normal(
    scale=np.sqrt(specific_var),
    size=(T, n),
)

returns = (
    factor_returns @ X.T
    + latent_true_returns[:, None] * y_true[None, :]
    + specific_returns
)

latent_est = estimate_latent_factor_returns(
    returns=returns,
    exposures=X,
    latent_factors=np.repeat(y_hat[None, :], T, axis=0),
    weights=weights,
)

nu_hat = estimate_latent_variance(latent_est.latent_returns)

print("\nlatent return estimation:")
print("true nu:", nu_true)
print("estimated nu:", nu_hat)
print("corr estimated vs true:",
      float(np.corrcoef(latent_est.latent_returns, latent_true_returns)[0, 1]))


base risk:
base model variance: 0.10920446
base true variance:  0.13267366
base exposure y_true: -0.04844502400569143
missing variance: 0.023469203509120188

latent return estimation:
true nu: 10.0
estimated nu: 6.412436015315739
corr estimated vs true: 0.9669962228497528


The AAF covariance matrix is

$$
Q_{\text{AAF}}
=
Q_{\text{model}}
+
\hat{\nu} \hat y \hat y^\top.
$$

We then solve the same optimization problem again, but with $Q_{\text{AAF}}$. The constraints and alpha are unchanged. Only the risk model changes. The intended effect is that the optimizer now recognizes the hidden implied-alpha direction as risky. It should therefore reduce exposure to $y_{\text{hat}}$ and, if the estimate is good, also reduce exposure to $y_{\text{true}}$.

In [146]:
aaf = solve_aaf_mvo(
    alpha=alpha,
    covariance=Q_model,
    constraints=constraints,
    risk_aversion=risk_aversion,
    latent_factor=y_hat,
    latent_variance=nu_hat,
)

h_aaf = aaf.holdings

Q_aaf = augment_covariance(
    covariance=Q_model,
    latent_factor=y_hat,
    latent_variance=nu_hat,
)

print("\nlatent exposure:")
print("base y_true:", float(h_base @ y_true))
print("AAF  y_true:", float(h_aaf @ y_true))
print("base y_hat: ", float(h_base @ y_hat))
print("AAF  y_hat: ", float(h_aaf @ y_hat))

print("\nvariance comparison:")
print(f"base model variance: {var(h_base, Q_model):.8f}")
print(f"base true variance:  {var(h_base, Q_true):.8f}")
print(f"AAF aug variance:    {var(h_aaf, Q_aaf):.8f}")
print(f"AAF true variance:   {var(h_aaf, Q_true):.8f}")

print("\nportfolio comparison:")
print("base alpha:", float(alpha @ h_base))
print("AAF alpha: ", float(alpha @ h_aaf))
print("L1 change:", float(np.sum(np.abs(h_aaf - h_base))))


latent exposure:
base y_true: -0.04844502400569143
AAF  y_true: 0.0006780413396226675
base y_hat:  -0.04847055666693667
AAF  y_hat:  -0.0052656232019106936

variance comparison:
base model variance: 0.10920446
base true variance:  0.13267366
AAF aug variance:    0.10221678
AAF true variance:   0.10204358

portfolio comparison:
base alpha: 0.07028268676511232
AAF alpha:  0.061602969958269346
L1 change: 0.13670899738612158


The final output compares the base portfolio and the AAF portfolio.

The base portfolio has higher expected alpha, but it also has hidden risk exposure. The base model does not see this risk, so

$$
h_{\text{base}}^\top Q_{\text{model}}h_{\text{base}}
<
h_{\text{base}}^\top Q_{\text{true}}h_{\text{base}}.
$$

The AAF portfolio gives up some alpha, but it reduces exposure to the hidden factor. Its augmented model variance is close to its true variance:

$$
h_{\text{AAF}}^\top Q_{\text{AAF}}h_{\text{AAF}}
\approx
h_{\text{AAF}}^\top Q_{\text{true}}h_{\text{AAF}}.
$$

So the example shows the core point:

**A portfolio can have no raw alpha-risk misalignment and still suffer from constraint-induced misalignment. The optimizer's binding constraints can create an orthogonal implied-alpha component. If that component has systematic risk, the base model underestimates risk. AAF adds that direction to the risk model and produces a more reliable optimization.**

## References

Saxena, A., Martin, C., and Stubbs, R. A. (2013). *Constraints in quantitative strategies: An alignment perspective*. Journal of Asset Management, 14, 278–292.

Ceria, S., Saxena, A., and Stubbs, R. A. (2012). *Factor Alignment Problems and Quantitative Portfolio Management*. The Journal of Portfolio Management, 38(2), 29–43.